# Before you model: what a data set can support

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/06_before_you_model.ipynb)

The first five notebooks build models and use them. Before a model is built, a few checks
decide whether it is worth building. Related samples are held out together. The model is
compared with counts of atoms or with one number. The question is asked whether a frame adds
anything. Every reference is given a search of its own settings. The notebooks after this one
make these checks by hand, each on its own kind of sample.

This notebook is about the tool that makes these checks for a table of molecules and activities,
before any model is built: the function `diagnose`, which is `cmf diagnose` on the command line.
It is not a model and it does not improve one. It answers five questions.

1. **Is there anything to model?** The number of molecules and the spread of the activity.
2. **Can a frame exist?** How many different scaffolds the set has, and whether the molecules
   are superimposed.
3. **Does the frame carry signal?** The model in the frame of the molecules against two routes
   that need no frame: counts of atom types, and the rotation-invariant kernel of the fields.
4. **Does the geometry add anything?** The same model against a few descriptors that need no
   three-dimensional structure.
5. **Was the model specified sensibly?** Whether the chosen attenuation lies at the edge of the
   values that were tried.

Then it recommends a route. The notebook runs it on four sets from the repository whose answers
differ, reads one report line by line, and ends with what the recommendation is worth.

A last section asks the third question without fitting any model: a test of whether the activity
depends on the fields beyond a count of atoms, `cmf dependence`.

Running time: about two minutes.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## One report, read line by line

The first set is the thrombin series of the Sutherland collection: 88 inhibitors, superimposed
by the authors of the set, with a measured pK<sub>i</sub>. The charges are read from the file.

In [2]:
import json
import numpy as np
from rdkit import RDLogger
from cmfqsar import load_dataset
from cmfqsar.diagnose import diagnose

RDLogger.DisableLog("rdApp.*")
ROUTES = ("aligned", "counts", "invariant", "descriptors")       # the routes of `cmf diagnose`


def ask(path, charges):
    """The diagnosis of one file, as the command line runs it."""
    data = load_dataset(path)
    y = np.asarray(data.activity, float)
    return diagnose(data.mols, data.names, y, charge_model=charges, routes=ROUTES,
                    activity_prop=data.activity_prop), data, y


reports = {}
reports["thrombin"], thrombin, y_thrombin = ask("examples/sutherland/THR.sdf", "file")
print(reports["thrombin"].report())

88 molecules, activity from 4.36 to 8.48 (sd 1.01)

Hyper-parameters chosen inside each fold and scored outside it.

How well each route predicts, 10-fold CV:
  aligned      q2 +0.616  RMSEcv 0.624  (alpha 0.4, lambda 0.1)
  counts       q2 +0.051  RMSEcv 0.980  (lambda 0.1)
  invariant    q2 +0.511  RMSEcv 0.703  (lambda 0.1)   width of its distance axis 0.75 A, not searched
  descriptors  q2 +0.057  RMSEcv 0.977  (lambda 1)   4 descriptors

What the set says:
! 40 chemotype(s); the largest holds 16 of 88 molecules -- a series of several scaffolds has no common frame, and an alignment will invent one
- the common framework (framework of the largest chemotype) has 30 heavy atoms, about 83% of a typical molecule, and 25% of the set can be cut at it
  the 30 shared framework atoms sit 0.0 A apart between molecules, over the best of their matches (22 of 88 carry them)
  the aligned model is +0.564 in q2 against counts (95% interval +0.366 to +0.810) -- the frame does better than counts
! 

The report has three parts.

**How well each route predicts.** Four ways of comparing the molecules, each with a kernel ridge
regression and a ten-fold cross-validation. The settings of every route are chosen inside each
training part, so the numbers are comparable although the routes have different numbers of
settings.

- `aligned` is the ordinary model of the first notebook: five fields in the frame in which the
  molecules arrive.
- `counts` knows how many atoms of each type a molecule has, and nothing about its geometry.
- `invariant` is the rotation-invariant kernel of the second notebook: the same fields, compared
  through the distances between atoms, so no frame is needed.
- `descriptors` is a regression on a few numbers that need no three-dimensional structure, such
  as the molecular weight and counts of hydrogen-bond donors and acceptors.

**What the set says.** One line for every finding. A line that starts with `!` is a warning, a
line with `-` a note. For thrombin there are two warnings, and they show how to read them.

- *Forty chemotypes.* The program could not have aligned this set by itself, because there is no
  common scaffold. The next line says that the molecules are superimposed all the same: the
  shared atoms lie 0.0 Å apart. So a frame exists. The authors made it, by docking.
- *The frame gains nothing over the invariant route.* The aligned model is ahead by 0.10, and the
  interval of that difference reaches just below zero. With 88 molecules a difference of 0.10 is
  at the limit of what can be seen.

The other two comparisons are clear: the frame is far ahead of the counts and of the
descriptors, with intervals well above zero. So the selectivity of thrombin for these inhibitors
depends on where the groups are, and not only on which groups there are.

**The recommendation** follows from these lines: the aligned route, because the frame carries
signal and the molecules are superimposed. It is a prompt and not a verdict. It says which route
to take, and it never says that a model is good.

## Four sets, four answers

The same function on three more sets. Two are from the same collection, inhibitors of the
angiotensin-converting enzyme (ACE) and ligands of the benzodiazepine receptor (BZR). The fourth
is a table of corrosion inhibitors with one heterocycle in common, given as SMILES, so the
program has to make the three-dimensional structures itself.

In [3]:
reports["ACE"], ace, y_ace = ask("examples/sutherland/ACE.sdf", "file")
reports["benzodiazepine receptor"], bzr, y_bzr = ask("examples/sutherland/BZR.sdf", "file")
reports["corrosion, thiadiazoles"], corrosion, y_corrosion = ask("examples/corrosion/7_c1nncs1.csv", "gasteiger")

print("%-26s %4s %9s %8s %10s %12s %14s   %s" % ("set", "n", "aligned", "counts", "invariant", "descriptors", "file columns", "recommended"))
for name, d in reports.items():
    q2 = [d.routes[r].q2 for r in ROUTES]
    own = "%+.2f" % d.routes["file columns"].q2 if "file columns" in d.routes else "none"
    print("%-26s %4d %+9.2f %+8.2f %+10.2f %+12.2f %14s   %s" % ((name, d.n) + tuple(q2) + (own, d.recommended)))

set                           n   aligned   counts  invariant  descriptors   file columns   recommended
thrombin                     88     +0.62    +0.05      +0.51        +0.06           none   aligned, with the frame checked
ACE                         114     +0.60    +0.66      +0.68        +0.64           none   descriptors
benzodiazepine receptor     147     +0.28    +0.20      +0.41        +0.13           none   invariant
corrosion, thiadiazoles      29     -0.09    -0.23      -0.19        -0.13          +0.99   none


Four sets and four different recommendations. Each is read below. The column `file columns`
is not a route to choose: it is explained with the fourth set.

### ACE: descriptors do as well

On this set all four routes reach about the same score, and the aligned model is the lowest of
them. The warnings say that the frame gains nothing over any other route.

In [4]:
for finding in reports["ACE"].findings:
    if finding.level == "warn" and "aligned model" in finding.text:
        print(finding)
print()
print("recommended:", reports["ACE"].recommended)
print("  ", reports["ACE"].why)
descriptors = reports["ACE"].routes["descriptors"]
print()
print("the descriptor route:", descriptors.note)
print("its descriptors with the highest correlation with the activity:",
      ", ".join("%s (r %+.2f)" % pair for pair in descriptors.correlations[:4]))

! the aligned model is -0.056 in q2 against counts (95% interval -0.179 to +0.052) -- the frame gains nothing over counts
! the aligned model is -0.072 in q2 against invariant (95% interval -0.202 to +0.020) -- the frame gains nothing over invariant
! the aligned model is -0.040 in q2 against descriptors (95% interval -0.170 to +0.065) -- descriptors alone do as well, so the geometry adds nothing on this set

recommended: descriptors
   descriptors alone match the 3D model, so a field model here is an expensive way to count atoms; use the fields only if the co-field is the purpose

the descriptor route: 4 descriptors
its descriptors with the highest correlation with the activity: calc:mw (r +0.76), calc:rotatable (r +0.61), calc:tpsa (r +0.54), calc:clogp (r +0.30)


The inhibitors of this set run from small dipeptide mimics to large molecules, and the activity
grows with the size. A count of atoms knows the size, so the three-dimensional model has nothing
to add in prediction. This does not make the field model useless here. Its co-field can still
be drawn and read, and the recommendation says so. But its score must not be presented as
evidence that the geometry matters.

### The benzodiazepine receptor: no frame needed

Here the route without a frame is the best one, and the aligned model is not ahead of it. The
recommendation is the invariant route, and its reason is practical: an alignment that gains
nothing is work that can be done wrongly.

One thing about this route has to be known. The invariant kernel has a setting of its own, the
width of its distance axis, and the diagnosis does not search it: it runs the default of 0.75 Å
and says so in the report. The catalyst series of the tenth notebook shows that such a setting
can decide a comparison. So it is tried here by hand.

In [5]:
from cmfqsar.diagnose import invariant_route

print("the aligned route on this set:      q2 %+.2f" % reports["benzodiazepine receptor"].routes["aligned"].q2)
for width in (0.25, 0.5, 0.75, 1.5):
    route = invariant_route(bzr.mols, bzr.names, y_bzr, charge_model="file", width=width)
    print("the invariant route, width %-5g A: q2 %+.2f" % (width, route.q2))

the aligned route on this set:      q2 +0.28


the invariant route, width 0.25  A: q2 +0.46


the invariant route, width 0.5   A: q2 +0.38
the invariant route, width 0.75  A: q2 +0.41


the invariant route, width 1.5   A: q2 +0.33


On this set the width changes the score a little, and no value changes the reading: the
invariant route stays at or above the aligned one. On molecules of this size the default is a
fair choice. On a series of much larger molecules it was not, and there a smaller width was far
better. When the recommendation depends on the comparison with this route, a few lines like
these are worth running.

### The thiadiazoles: nothing to model, and the columns of the file

The fourth set is the instructive one. It has 29 molecules, and no route that describes the
molecules predicts the activity. Still one number in its row of the table is 0.99.

In [6]:
d = reports["corrosion, thiadiazoles"]
for route in d.routes.values():
    print("%-13s q2 %+.2f   %s" % (route.name, route.q2, route.note))
print()
for finding in d.findings:
    if finding.code in ("superimposed", "file-columns"):
        print(finding)
        print()
print("recommended:", d.recommended)
print("  ", d.why)

aligned       q2 -0.09   
counts        q2 -0.23   
invariant     q2 -0.19   width of its distance axis 0.75 A, not searched
descriptors   q2 -0.13   4 descriptors
file columns  q2 +0.99   10 descriptors; dropped as the activity under another name: mwila_pooled_raw; not used for the recommendation

! the 20 shared framework atoms sit 1.3 A apart between molecules, over the best of their matches (3 of 29 carry them) -- the molecules are not in a common frame

! the file has 11 numeric column(s) of its own (mw, n_papers, dG_ads, depth, mwila_hcl_adj, mwila_h2so4_raw, mwila_h2so4_adj, mwila_pooled_raw and 3 more); with them the descriptor baseline is +1.123 in q2 against the computed descriptors (95% interval +0.955 to +1.547); the activity under another name, by a correlation above 0.95: mwila_pooled_raw -- these columns predict the activity, so they are measurements or forms of the activity and not descriptions of the structure. They are not used. If some of them are real descriptors, s

Two warnings carry the lesson.

- **The molecules are not in a common frame.** The file holds SMILES, the structures were
  generated one by one, and nothing superimposed them. An aligned model of such a set compares
  molecules in arbitrary orientations.
- **The columns of the file predict the activity.** The table has eleven numeric columns besides
  the activity. With them a descriptor baseline reaches a q² of 0.99, where the descriptors
  computed from the structure reach nothing. These columns are not descriptions of the
  molecules. One is the activity under another name, and others are adjusted forms of the same
  measurement, which rebuild it between them.

The diagnosis therefore does not use such columns. The route `descriptors` takes only what the
program computes from the structure. The columns of the file are scored beside it, as the route
`file columns`, which is printed and never recommended. If a column of a file is a real
descriptor, for example the temperature of the measurement, it is named, and then it joins the
baseline: `diagnose(..., descriptors=["T"])`, or `--descriptors T` on the command line.

The recommendation is `none`. That is a result. It says that this table, as it is, does not
support a model, and it says so in three seconds, before a week is spent on fields and frames.

### Why the columns are left out and not tested

An earlier form of the diagnosis took all columns of the file into the baseline and then looked
for a leak: it refused the baseline when it reached a q² of 0.95. That is a threshold, and a
threshold can be passed. The cell shows the case that passed it, a related table of corrosion
inhibitors with a pyrazole ring, which has the same columns. The diagnosis is run on all 25
molecules and on 17 of them.

In [7]:
pyrazoles = load_dataset("examples/corrosion/13_c1c_nH_nn1.csv")
y_pyrazoles = np.asarray(pyrazoles.activity, float)
part = sorted(np.random.RandomState(3).permutation(len(y_pyrazoles))[8:].tolist())      # 17 of the 25

print("%-18s %22s %14s   %s" % ("", "computed descriptors", "file columns", "recommended"))
for label, rows in (("all 25 molecules", list(range(len(y_pyrazoles)))), ("17 of them", part)):
    d = diagnose([pyrazoles.mols[i] for i in rows], [pyrazoles.names[i] for i in rows], y_pyrazoles[rows],
                 routes=ROUTES, activity_prop=pyrazoles.activity_prop)
    print("%-18s %+22.2f %+14.2f   %s" % (label, d.routes["descriptors"].q2, d.routes["file columns"].q2, d.recommended))

                     computed descriptors   file columns   recommended


all 25 molecules                    +0.35          +0.98   descriptors


17 of them                          +0.18          +0.94   invariant


On the 17 molecules the columns of the file give a q² of 0.94, a little below the old threshold.
The old diagnosis gave no warning there and recommended descriptors, which meant: predict the
activity from the activity. Now those columns do not enter the baseline, whatever they score,
so no threshold decides anything.

The two recommendations of this cell differ from each other, and that is a second lesson. With
17 to 25 molecules a recommendation is not stable; the report says how wide the intervals are.

## Searching harder is not a better model

Every route in the reports above chose its settings inside each training part. The option
`nested=False` switches that off: each route is then scored at the best point of its own grid,
which is how a quick comparison is often made. The cell runs the ACE set both ways.

In [8]:
flat = diagnose(ace.mols, ace.names, y_ace, charge_model="file", routes=ROUTES,
                activity_prop=ace.activity_prop, nested=False)
print("%-12s %28s %30s %12s" % ("route", "settings chosen inside", "best point of its own grid", "difference"))
for name in ROUTES:
    inside, best = reports["ACE"].routes[name].q2, flat.routes[name].q2
    print("%-12s %+28.3f %+30.3f %+12.3f" % (name, inside, best, best - inside))

route              settings chosen inside     best point of its own grid   difference
aligned                            +0.605                         +0.657       +0.052
counts                             +0.660                         +0.654       -0.007
invariant                          +0.677                         +0.680       +0.003
descriptors                        +0.644                         +0.633       -0.011


The aligned route gains most from being scored at its best point, because it searches the most
settings: six attenuations and five ridge parameters. The other routes search the ridge
parameter only, and they change by about 0.01. So a comparison at the best points favours the route with
the largest grid, and with it the aligned model would look better against its references than
it is. This is the same fault as a reference that is not searched, seen from the other side.

## What the advice is worth

A tool that reports four baselines is useful whatever its last line says. Whether the last
line, the recommendation, is good advice has to be measured. The repository has done that
(`python benchmarks/run_diagnose.py --validate`): 30 % of a set is held out, the diagnosis is
run on the other 70 %, the model that it recommends is built, and that model is scored on the
molecules that were held out, against every other route. Thirteen sets, three splits each. The
cell reads the stored results.

In [9]:
import glob
import sys

sys.path.insert(0, "benchmarks")
import run_diagnose                               # the benchmark script; it holds the summary

stored = []
for path in sorted(glob.glob("benchmarks/results/diagnose/*_validate.json")):
    with open(path, encoding="utf-8") as fh:
        stored.append(json.load(fh))
splits = run_diagnose.validation_splits(stored)
s = run_diagnose.advice_summary(splits)
print("%d splits of %d sets" % (s["splits"], s["sets"]))
print("the advised route was the best of the five in %d splits; it fell %.3f in R2 short of the best on the average" % (s["advice_best"], s["advice_cost"]))
print("the aligned route was the best in %d splits; always taking it costs %.3f" % (s["aligned_best"], s["aligned_cost"]))
print("the advice minus the aligned route, mean over the sets: %+.3f [%+.3f, %+.3f]" % (s["gain"], s["gain_low"], s["gain_high"]))
print("sets on which the advice is better: %d, worse: %d" % (s["sets_better"], s["sets_worse"]))
print("splits in which a route that reads the activity was advised:", sum(x["leak"] for x in splits))
print("splits in which the advice was that no route is worth a model:", sum(x["no_route"] for x in splits))

39 splits of 13 sets
the advised route was the best of the five in 16 splits; it fell 0.117 in R2 short of the best on the average
the aligned route was the best in 12 splits; always taking it costs 0.175
the advice minus the aligned route, mean over the sets: +0.058 [-0.048, +0.225]
sets on which the advice is better: 5, worse: 5
splits in which a route that reads the activity was advised: 0
splits in which the advice was that no route is worth a model: 3


The advice names the best route in fewer than half of the splits. Following it is better than
always taking the aligned model by 0.06 in R² on the average over the sets, and the interval of
that difference contains zero. It is better on five sets and worse on five.

So the recommendation is **insurance and not an advantage**. It costs a little on sets that are
well aligned, where the aligned model was right all along. It protects against the case in
which an automatic frame means nothing: on one corrosion table the aligned model falls to an R²
of -1.26 on held-out molecules, and the advised route reaches 0.57. It is not a way to a better
model on the average.

The line before the last belongs to the section on the columns of a file. With the earlier
form of the diagnosis five of these splits advised descriptors that were the activity in
another form, and their scores near 1 were counted as the largest successes of the advice. The
gain then looked twice as large as it is. The summary still looks for such a split, and it
finds none.

The last line counts the thiadiazoles. On each of their three splits the advice was that no
route is worth a model. The summary scores such a split as the aligned model, because that is
what a user who builds a model anyway would build. In two of the three no route reached an R² of
0.05 on the molecules that were held out.

## Beyond a count: a test instead of a model

The routes above answer "do the fields add anything" by fitting models and comparing their
errors. On the ACE set the activity grows with the size of the molecule, and a count of atoms
knows the size. So the natural question before a field model is built: **do the fields carry
any information about the activity beyond the number of heavy atoms?**

The function `hsic_test`, `cmf dependence` on the command line, asks this question without
fitting a model.

- **The statistic** is HSIC, the Hilbert-Schmidt independence criterion: a measure of the
  dependence between two variables computed from their kernel matrices. With the field kernel
  and the activity, it is the squared length of the field weighted by the centred activity. It
  is zero when the mean activity does not depend on the fields. It is reported as an
  *alignment* between 0 and 1, which does not grow with the number of molecules.
- **The p-value** comes from 999 random permutations of the activities over the molecules.
  Such a p-value is exact for any number of molecules: when there is no dependence, it falls
  below 0.05 in 5 % of the data sets.
- **Beyond a reference.** Given the number of heavy atoms, the molecules with the same number
  form a *stratum*. The activities are permuted only within a stratum, and both kernels are
  centred within the strata, so only the dependence among molecules of the same size counts.
  If the activity depends on the count alone, these permutations change nothing, and the test
  stays exact.

The reference for the test is the usual answer: a kernel ridge model on the count against a
model on the count and the fields, ten-fold cross-validation over five fold seeds, with a paired
bootstrap interval of the gain in q² over the molecules. The test is useful only if it agrees
with that comparison or does better.

In [10]:
import run_hsic                                  # the benchmark script; it holds the model comparison
from cmfqsar.hsic import hsic_test, strata_of

mols, y, K = run_hsic.load_set("ACE")             # the five fields, alpha 0.3, charges from the file
heavy = run_hsic.reference_values("heavy", mols)  # the number of heavy atoms of each molecule
strata, exact = strata_of(heavy)
print("%d molecules, %d different numbers of heavy atoms" % (len(y), strata.max() + 1))
print()
print("all fields:      ", hsic_test(K, y, permutations=999).describe())
print("beyond the count:", hsic_test(K, y, strata=strata, exact=exact, permutations=999).describe())
mc = run_hsic.model_comparison(K, heavy, y, seeds=(0, 1, 2, 3, 4))
print()
print("the model comparison: the fields added to a model on the count gain %+.3f in q2 [%+.3f, %+.3f]"
      % (mc["gain_q2"], mc["low"], mc["high"]))

114 molecules, 29 different numbers of heavy atoms

all fields:       dependence of the activity on the fields: alignment 0.253, p = 0.001 from 999 permutations (exact), z = +18.5
beyond the count: dependence within 29 strata of the reference (108 of 114 molecules share a stratum): alignment 0.095, p = 0.012 from 999 permutations (exact), z = +3.4

the model comparison: the fields added to a model on the count gain +0.106 in q2 [+0.021, +0.192]


The first line answers a question nobody asked. The activity depends on the fields (p = 0.001,
the smallest value 999 permutations can give), but the fields see the size of a molecule, and
the activity grows with the size. So this line cannot separate the fields from a count.

The second line is the answer. Among molecules with the same number of heavy atoms, the
activity still depends on the fields: p = 0.012, with an alignment of 0.095, smaller than the
0.253 of all fields. 108 of the 114 molecules share their number of heavy atoms with another
molecule, so almost all of them count. The model comparison agrees: the fields added to a model
on the count gain 0.106 in q², and the interval excludes zero.

This does not contradict the diagnosis above, which found that descriptors do as well as the
aligned model. The descriptor route uses several numbers, and the test here is given only one.

In [11]:
import time

g = run_hsic.standard(run_hsic.g(run_hsic.standard(heavy)))   # a curve in the count, half of the variance
signal = run_hsic.field_direction(K, strata)                   # a field direction the count does not explain


def rejected(y_rep, seed):
    """Is each test positive at the 5 % level?"""
    return {"test, all fields": hsic_test(K, y_rep, permutations=199, seed=seed).p_value <= 0.05,
            "test on the residual of a line": hsic_test(K, run_hsic.residual_on_line(y_rep, heavy),
                                                        permutations=199, seed=seed).p_value <= 0.05,
            "test beyond the count": hsic_test(K, y_rep, strata=strata, permutations=199,
                                               seed=seed).p_value <= 0.05}


start = time.time()
print("%-32s %26s %26s" % ("", "no signal beyond the count", "5 % of the variance from it"))
rates = {}
for share, replicates in ((0.0, 1000), (0.05, 200)):
    rng = np.random.RandomState(99 if share == 0 else 105)
    counts = None
    for t in range(replicates):
        r = rejected(run_hsic.synthetic(rng, g, signal, share), 10000 + t)
        counts = {k: counts[k] + v for k, v in r.items()} if counts else {k: int(v) for k, v in r.items()}
    models = [run_hsic.model_comparison(K, heavy, run_hsic.synthetic(rng, g, signal, share), seed=t)["positive"]
              for t in range(40)]
    rates[share] = {k: v / replicates for k, v in counts.items()}
    rates[share]["model comparison"] = float(np.mean(models))
    rates[share]["replicates"] = replicates
for rule in rates[0.0]:
    if rule != "replicates":
        print("%-32s %25.1f%% %25.1f%%" % (rule, 100 * rates[0.0][rule], 100 * rates[0.05][rule]))
se = np.sqrt(0.05 * 0.95 / rates[0.0]["replicates"])
print()
print("a test that is right rejects 5 %% of the activities without a signal: %.1f to %.1f %% by chance with %d replicates"
      % (100 * (0.05 - 2 * se), 100 * (0.05 + 2 * se), rates[0.0]["replicates"]))
print("(the model comparison on 40 activities of each kind; %.0f s)" % (time.time() - start))

                                 no signal beyond the count 5 % of the variance from it


test, all fields                                     100.0%                     100.0%
test on the residual of a line                         4.5%                      75.0%
test beyond the count                                  5.1%                      91.5%
model comparison                                       0.0%                       0.0%

a test that is right rejects 5 % of the activities without a signal: 3.6 to 6.4 % by chance with 1000 replicates
(the model comparison on 40 activities of each kind; 35 s)


These are activities made on the real ACE molecules, so that the answer is known. Without a
signal, the activity is a curve in the number of heavy atoms plus noise, and the curve carries
half of the variance. With a signal, a field direction that the count does not explain is
added, and it carries 5 % of the variance. That direction is the leading one of the kernel
within the strata, which is the direction the test sees best. A signal spread over many
directions would be found less often.

- **The test on all fields is useless for this question.** It rejects every activity that
  depends on the count alone, because the fields see the size.
- **The usual quick repair**, the test on the residual of a straight line in the count, is
  right on ACE (4.5 %). The table of all eight sets below shows that this is luck.
- **The test beyond the count** rejects 5.1 % of the activities without a signal, inside the
  range that chance allows. With the signal it finds it in 91.5 % of the activities.
- **The model comparison** never finds the signal (0 of 40). It never gives a false positive
  either. It answers a harder question: whether the signal can be predicted on new molecules.
  A signal with 5 % of the variance cannot, on 114 molecules.

So for this question the test is far more sensitive than a comparison of models, and it is
still right when there is nothing to find.

In [12]:
with open("benchmarks/results/hsic/calibration.json", encoding="utf-8") as fh:
    calibration = json.load(fh)["sets"]
with open("benchmarks/results/hsic/real.json", encoding="utf-8") as fh:
    real = json.load(fh)["sets"]

print("synthetic activities on the molecules of each set, given the number of heavy atoms;")
print("rejections at the 5 % level, 400 activities of each kind (the model comparison: 40, on sets of at most 150 molecules)")
print()
print("%-6s %4s %20s %20s %20s %20s" % ("set", "n", "no signal: residual", "no signal: test", "5 %: test", "5 %: models"))
for d in calibration:
    sh = d["references"]["heavy atoms"]["shares"]
    mod = sh["0.05"]["model comparison"]
    print("%-6s %4d %19.1f%% %19.1f%% %19.1f%% %20s" % (d["set"], d["n"], 100 * sh["0.0"]["residual"],
          100 * sh["0.0"]["conditional"], 100 * sh["0.05"]["conditional"], "--" if mod is None else "%.1f%%" % (100 * mod)))

print()
print("the measured activities, given the number of heavy atoms")
print("%-6s %4s %8s %16s %32s" % ("set", "n", "strata", "p, beyond count", "model comparison: gain in q2"))
for d in real:
    r = d["references"]["heavy atoms"]
    m = r["model comparison"]
    print("%-6s %4d %8d %16.3f %+12.3f [%+.3f, %+.3f]" % (d["set"], d["n"], r["conditional"]["strata"],
          r["conditional"]["p_value"], m["gain_q2"], m["low"], m["high"]))

synthetic activities on the molecules of each set, given the number of heavy atoms;
rejections at the 5 % level, 400 activities of each kind (the model comparison: 40, on sets of at most 150 molecules)

set       n  no signal: residual      no signal: test            5 %: test          5 %: models
ACE     114                 3.8%                 7.5%                91.2%                 0.0%
AChE    111                12.2%                 4.5%                81.5%                 0.0%
BZR     147                 9.0%                 6.5%                95.5%                 7.5%
COX2    282                 8.2%                 4.8%                99.8%                   --
DHFR    361               100.0%                 4.0%               100.0%                   --
GPB      66                17.2%                 3.8%                59.0%                 0.0%
THER     76                13.8%                 4.2%                70.8%                 0.0%
THR      88                56

The same activities were made on all eight Sutherland sets (`benchmarks/run_hsic.py`, 400 of
each kind). The cell reads the stored results.

- **The residual of a line is not a repair.** Without a signal it rejects 4 to 17 % of the
  activities on six sets, 56 % on THR and all of them on DHFR. A straight line removes only the
  straight part of the curve, and how much is left depends on the set.
- **The test beyond the count stays near 5 %** on every set, from 3.8 to 7.5 %, and 5.2 % over
  all sets. On ACE the stored 400 activities gave 7.5 %, against 5.1 % on the 1,000 above, which
  used other random numbers. For a right test, at least one of eight sets reaches 7.5 % in about
  one case in seven, so this is chance.
- **Its power depends on the set:** 59 to 100 % for a signal with 5 % of the variance. The model
  comparison found that signal in at most 7.5 % of the activities.

On the measured activities the model comparison finds a gain from the fields beyond the count on
all eight sets. The test agrees on seven. The exception is THER (p = 0.42, against a gain of
0.16 [0.02, 0.29]). THER has 76 molecules in 28 strata, and twelve of them are alone in their
stratum, so the test has little to work with. A count with many values on a small set is the
weak case of the test, and a large p-value there is not evidence that the fields add nothing.

Two more results are in the manual and in `benchmarks/results/hsic/table.md`:

- **A continuous reference**, such as logP, has no strata and must be cut into bins. Bins alone
  are not enough, because the activity still depends on logP inside a bin: the test then
  rejected 19 to 22 % of the activities without a signal. The command therefore also removes a
  straight line in the reference within each bin (`--bins`). That brought the rate to 1.5 to
  2.4 %, but the test is then approximate and not exact.
- **A case with a known answer.** On the 1,048 ethers of the ion-and-solvent notebook, the
  fields predict the binding energy of Li+, Na+ and Mg2+ better than a count of the oxygens that
  touch the ion. Given that count, the test finds information in the fields for all three ions.
  With an activity made from the count alone, it rejects 1 of 100.

In [13]:
import subprocess

run = subprocess.run([sys.executable, "cmf_cli.py", "dependence", "examples/sutherland/ACE.sdf",
                      "--charges", "file", "--alpha", "0.3", "--given", "calc:HeavyAtomCount"],
                     capture_output=True, text=True)
print(run.stdout)

114 molecules; fields Electrostatic, Steric, Hydrophobic, H-bond acceptor, H-bond donor, alpha = 0.3; 999 permutations, linear kernel on the activity

Does the activity depend on the fields at all?
  dependence of the activity on the fields: alignment 0.253, p = 0.001 from 999 permutations (exact), z = +18.5

Does it depend on the fields beyond calc:HeavyAtomCount?
  dependence within 29 strata of the reference (108 of 114 molecules share a stratum): alignment 0.095, p = 0.012 from 999 permutations (exact), z = +3.4
  the fields carry information about the activity that calc:HeavyAtomCount does not



The command gives the same numbers as the function, because `--charges file --alpha 0.3` asks
for the same kernel. The reference is any property of the file, or any descriptor RDKit
computes, written `calc:` and its name. The default attenuation of the command is 0.5.

**What the test does not say.** A small p-value says that the fields carry information about
the activity that the count does not. It does not say that a field model will predict that
information on new molecules, which is what the model comparison asks. The test is a check
before a model is built: if it finds nothing beyond a count on a set of reasonable size, a field
model on that set may be measuring the count in another form.

## What to take from this notebook

- Run the diagnosis before the model. It takes seconds to a few minutes.
- Read the lines and not only the last one. A warning is a question that the set has not
  answered.
- `aligned` is recommended only when the frame carries signal and the molecules are really
  superimposed. `invariant` often means that the frame is not worth the work, not that the
  invariant kernel predicts better. `descriptors` and `counts` mean that the geometry adds
  nothing to the prediction. `none` means that the table does not support a model.
- The columns of a file are not descriptors until you name them. They can be the activity in
  another form, and no test of a score can tell.
- Compare routes with their settings chosen inside the folds, and give a reference route the
  settings it needs. The diagnosis does the first, and for the invariant route the second is
  left to you.
- With twenty molecules a recommendation can change when a few of them are left out.
- To ask whether the fields carry anything beyond a count, test it with `cmf dependence --given`.
  For a count its false-positive rate is exact, and it finds a small signal that a comparison of
  models misses. A small p-value says that the information is there, not that a model will
  predict it.

From the command line:

```
cmf diagnose examples/sutherland/THR.sdf --charges file
cmf diagnose examples/corrosion/7_c1nncs1.csv --json diagnosis.json
cmf dependence examples/sutherland/ACE.sdf --charges file --given calc:HeavyAtomCount
```

Where to read further in the manual: the chapter *Before you model: `cmf diagnose`* has the five
questions with the figure of the decisions, the two protections against a leak and against a
search that is too hard, and the measurement of the advice. *Beyond a count: the conditional dependence
test* has the test and its measurement on the eight Sutherland sets. *Three tests to run before you build
a new field or a new score* asks the same kind of question of a new idea and not of a data set.